# 02 Validation

This notebook runs read-only checks on the local DuckDB warehouse before the analysis stage.

The goal is simple: confirm that the core tables exist, the main identifiers are unique where expected, and the remaining data issues are visible before interpreting the charts.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ROOT


In [ ]:
import duckdb
import pandas as pd
from IPython.display import Markdown, display

In [ ]:
DB_PATH = ROOT / "data" / "nba_analytics.duckdb"

if not DB_PATH.exists():
    raise FileNotFoundError(f"{DB_PATH} does not exist. Run 01_build_database.ipynb first.")

con = duckdb.connect(str(DB_PATH))
con.execute("SELECT current_database() AS database_name").fetchdf()

In [ ]:
queries = [
    (
        "1. Confirm expected tables exist",
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'main'
          AND table_name IN (
              'team_stats_raw',
              'player_stats_raw',
              'players_raw',
              'players',
              'teams',
              'games',
              'dates',
              'player_stats_fact',
              'team_stats_fact'
          )
        ORDER BY table_name
        """,
    ),
    (
        "2. Duplicate personId check in players dimension",
        """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT personId) AS distinct_person_ids,
            COUNT(*) - COUNT(DISTINCT personId) AS duplicate_person_id_rows
        FROM players
        """,
    ),
    (
        "3. Duplicate gameId check in games dimension",
        """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT gameId) AS distinct_game_ids,
            COUNT(*) - COUNT(DISTINCT gameId) AS duplicate_game_id_rows
        FROM games
        """,
    ),
    (
        "4. Team dimension ambiguity by name",
        """
        SELECT
            teamName,
            COUNT(*) AS dimension_rows,
            COUNT(DISTINCT teamCity) AS city_count,
            COUNT(DISTINCT coachId) AS coach_count
        FROM teams
        GROUP BY teamName
        HAVING COUNT(*) > 1
        ORDER BY dimension_rows DESC, teamName
        """,
    ),
    (
        "5. Unresolved foreign-key-style fields in player fact table",
        """
        SELECT
            SUM(CASE WHEN playerKey IS NULL THEN 1 ELSE 0 END) AS missing_playerkey,
            SUM(CASE WHEN teamKey IS NULL THEN 1 ELSE 0 END) AS missing_teamkey,
            SUM(CASE WHEN opponentTeamKey IS NULL THEN 1 ELSE 0 END) AS missing_opponentteamkey,
            SUM(CASE WHEN gameKey IS NULL THEN 1 ELSE 0 END) AS missing_gamekey,
            SUM(CASE WHEN dateKey IS NULL THEN 1 ELSE 0 END) AS missing_datekey
        FROM player_stats_fact
        """,
    ),
    (
        "6. Unresolved foreign-key-style fields in team fact table",
        """
        SELECT
            SUM(CASE WHEN teamKey IS NULL THEN 1 ELSE 0 END) AS missing_teamkey,
            SUM(CASE WHEN opponentTeamKey IS NULL THEN 1 ELSE 0 END) AS missing_opponentteamkey,
            SUM(CASE WHEN gameKey IS NULL THEN 1 ELSE 0 END) AS missing_gamekey,
            SUM(CASE WHEN dateKey IS NULL THEN 1 ELSE 0 END) AS missing_datekey
        FROM team_stats_fact
        """,
    ),
    (
        "7. Position-flag distribution in players dimension",
        """
        SELECT
            guard,
            forward,
            center,
            COUNT(*) AS players
        FROM players
        GROUP BY guard, forward, center
        ORDER BY players DESC, guard, forward, center
        """,
    ),
    (
        "8. Active players in 2024 or 2025 with multiple position flags",
        """
        SELECT
            COUNT(DISTINCT p.playerKey) AS active_multi_position_players
        FROM players p
        JOIN player_stats_fact ps ON ps.playerKey = p.playerKey
        JOIN dates d ON d.dateKey = ps.dateKey
        WHERE d.year IN (2024, 2025)
          AND COALESCE(p.guard, 0) + COALESCE(p.forward, 0) + COALESCE(p.center, 0) > 1
        """,
    ),
]

pd.DataFrame({"validation_check": [title for title, _ in queries]})

In [ ]:
results = {}

for title, sql in queries:
    display(Markdown(f"### {title}"))
    df = con.execute(sql).fetchdf()
    results[title] = df
    display(df)

## Reading The Results

- duplicate `gameId` rows would mean the game dimension still needs deduplication
- non-zero missing key counts indicate unresolved fact-to-dimension links
- multiple rows for the same `teamName` would point to team dimension ambiguity
- position-flag overlap shows why the final player benchmark does not rely on the raw Guard / Forward / Center fields

The analysis notebook avoids that issue by inferring player roles from box-score production instead.


In [ ]:
con.close()
